### 1. Advanced Embedding Setup
We use the **BGE-Base** model to ensure that nuanced legal terms (like "Self-Incrimination" vs "Confession") are mapped correctly in the vector space. This matches the professional standard used in the Lawglance architecture.

In [1]:
import os
import torch
import re
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import chromadb

load_dotenv()

# GPU check for BGE-Base model
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Using Device: {device}")

# We use BGE-Base for superior semantic mapping of legal terminology
model_name = "BAAI/bge-base-en-v1.5"
embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs={'device': device},
    encode_kwargs={'normalize_embeddings': True}
)

PERSIST_DIRECTORY = r"C:\Users\user\Desktop\RAG Projects\Legal RAG 1\vector_database"
client = chromadb.PersistentClient(path=PERSIST_DIRECTORY)

✅ Using Device: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### 2. Legal Document Cleaning
Before processing, we strip out non-ASCII characters and horizontal footer lines. This prevents the AI from citing "iLovePDF" stamps or page numbers as actual legal text, a common failure in basic RAG systems.

In [2]:
from langchain_community.document_loaders import PyMuPDFLoader

# You can now feed any legal PDF here
file_path = r"C:\Users\user\Desktop\RAG Projects\Legal RAG 1\data\Indian_Constitution.pdf" 
loader = PyMuPDFLoader(file_path)
raw_data = loader.load()

# Unified Cleaning: Remove non-ASCII (Hindi noise) and footers
for page in raw_data:
    page.page_content = "".join(i for i in page.page_content if ord(i) < 128)
    page.page_content = re.split(r'_{2,}', page.page_content)[0].strip()

print(f"✅ Ingested and cleaned {len(raw_data)} pages.")

✅ Ingested and cleaned 402 pages.


### 3. Context-Aware Hybrid Chunking

By prepending `LAW_SECTION_REFERENCE` to every chunk, we ensure the agent doesn't just summarize; it identifies which specific law is applicable even if the chunk itself is small.

In [3]:
import re
from langchain_experimental.text_splitter import SemanticChunker
from langchain_core.documents import Document

# 1. Consolidate your specific slices (From your code)
# This keeps your exact document structure for Preamble, Articles, and Schedules
articles_short = raw_data[3:31]
preamble = [raw_data[31]]
articles = raw_data[32:283]
schedules = raw_data[283:381]
all_pages = articles_short + preamble + articles + schedules

# 2. Unified Cleaning (Lawglance standard)
cleaned_docs = []
for doc in all_pages:
    # Remove Hindi/Non-ASCII noise to optimize BGE-Base embeddings
    text = "".join(i for i in doc.page_content if ord(i) < 128)
    # Remove underscores and footer stamps
    text = re.split(r'_{2,}', text)[0].strip()
    doc.page_content = text
    cleaned_docs.append(doc)

# 3. Hybrid Tagging & Semantic Chunking
# Using BGE-Base here ensures natural legal breakpoints
semantic_splitter = SemanticChunker(embeddings, breakpoint_threshold_type="percentile")
hybrid_docs = []

for doc in cleaned_docs:
    # Identify the 'Heading' (e.g., Article 21) to use as a Reference
    lines = doc.page_content.split('\n')
    page_reference = lines[0][:60].strip() if lines else "General Provision"
    
    # Split this specific page semantically
    sub_chunks = semantic_splitter.split_documents([doc])
    
    for chunk in sub_chunks:
        # THE HYBRID ENGINE: Prepend the reference directly into the text
        # This makes the metadata 'unbreakable' even if retrieved out of order
        tagged_content = f"LAW_SECTION_REFERENCE: {page_reference}\n\nCONTENT: {chunk.page_content}"
        
        hybrid_docs.append(Document(
            page_content=tagged_content,
            metadata=doc.metadata 
        ))

print(f"✅ Processed {len(hybrid_docs)} Hybrid Tagged Chunks.")

✅ Processed 787 Hybrid Tagged Chunks.


### 4. Vector Store Creation
We now take our tagged hybrid documents and store them in ChromaDB. By using `from_documents`, the `bge-base` model generates high-fidelity vectors for each chunk. These are saved to your local disk for use in the CrewAI chatbot.

In [4]:
# --- STEP 4: VECTOR STORE PERSISTENCE ---

# 1. CLEANING THE DATABASE
# To prevent "Vector Pollution" (mixing old small-model vectors with new base-model vectors),
# we must delete the previous collection.
try:
    client.delete_collection("legal_knowledge")
    print("🗑️ Database Purged: Cleared existing 'legal_knowledge' collection.")
except Exception as e:
    print(f"ℹ️ Note: No existing collection found or could not delete: {e}")

# 2. INITIALIZING THE HYBRID VECTOR STORE
# This uses the high-fidelity BGE-Base embeddings and our hybrid-tagged documents.
vector_store = Chroma.from_documents(
    documents=hybrid_docs, # Using the tagged documents from the previous cell
    embedding=embeddings,
    client=client,
    collection_name="legal_knowledge"
)

# 3. VERIFICATION
# Confirming that the exact number of hybrid chunks created has been saved.
total_chunks = vector_store._collection.count()
print(f"✅ SUCCESS: Persisted {total_chunks} hybrid context-aware chunks to disk.")
print(f"📍 Database Location: {PERSIST_DIRECTORY}")

ℹ️ Note: No existing collection found or could not delete: Collection [legal_knowledge] does not exists
✅ SUCCESS: Persisted 787 hybrid context-aware chunks to disk.
📍 Database Location: C:\Users\user\Desktop\RAG Projects\Legal RAG 1\vector_database


### 5. Retrieval Verification
Finally, we run a test query. We are checking to see if the retrieved content starts with the `REFERENCE` tag. This confirms that our "Context-Aware" RAG is working and the LLM will receive the Article number along with the text.

In [5]:
# --- STEP 5: HYBRID RETRIEVAL VERIFICATION ---

# Define a complex query to test if the "Strategist" logic in the next notebook will succeed
test_query = "What are the constitutional protections against self-incrimination?"

# We retrieve the top 3 results to inspect the 'Hybrid' tagging
print(f"🔍 Testing Retrieval for: '{test_query}'\n")
results = vector_store.similarity_search(test_query, k=3)

if not results:
    print("❌ ERROR: No results retrieved. Check your vector store initialization.")
else:
    for i, res in enumerate(results):
        print(f"--- [RESULT {i+1}] ---")
        # Check for our LAW_SECTION_REFERENCE tag at the start
        print(f"📄 CONTENT PREVIEW:\n{res.page_content[:500]}...") 
        print(f"\n📍 METADATA: {res.metadata}")
        print("-" * 50)

🔍 Testing Retrieval for: 'What are the constitutional protections against self-incrimination?'

--- [RESULT 1] ---
📄 CONTENT PREVIEW:
LAW_SECTION_REFERENCE: THE CONSTITUTION OF  INDIA

CONTENT: (2) No person shall be prosecuted and punished for the same offence 
more than once. (3) No person accused of any offence shall be compelled to be a witness 
against himself. 21. Protection of life and personal liberty.No person shall be 
deprived of his life or personal liberty except according to procedure 
established by law. 2[21A. Right to education.The State shall provide free and 
compulsory education to all children of the age o...

📍 METADATA: {'author': '', 'title': '', 'creationDate': '', 'page': 41, 'format': 'PDF 1.7', 'keywords': '', 'total_pages': 402, 'file_path': 'C:\\Users\\user\\Desktop\\RAG Projects\\Legal RAG 1\\data\\Indian_Constitution.pdf', 'creator': '', 'modDate': 'D:20240701112033Z', 'producer': 'iLovePDF', 'subject': '', 'moddate': '2024-07-01T11:20:33+00:00', 'trappe